# 第 8 周：多智能体淘便宜货系统

## 练习目标（理念）

用 **Mock** 版多智能体框架演示第 8 周「淘便宜货」产品形态：扫描 → 集成估价 → 超阈值告警，并配上带线程/队列的实时 **Gradio** UI 与 3D 向量空间占位图。

本笔记本是**演示/教学向 Mock**：不调真实 Modal 定价服务，也不连真实 ChromaDB；接口形状贴近课程完整版，方便本地无密钥也能跑通 UI。

## 和第 8 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多代理专长分工 | MockScanner / MockEnsemble / MockMessaging + Planning |
| 实时 Gradio UI | `gr.Timer` + 后台线程 + `queue` 拉日志 |
| 记忆与阈值 | `mock_memory.json` + `DEAL_THRESHOLD` |
| RAG / 向量空间（演示） | `get_plot_data` 随机点 + Plotly 3D |

## 怎么跑

1. 安装依赖格 → 导入 → 定义 Mock 代理与 Framework → UI 辅助函数 → `DealHunterApp` → `app.launch()`
2. 浏览器打开 Gradio；定时器会周期性触发扫描并把日志刷到页面


In [ ]:
# ========== 安装依赖：Gradio UI、Pydantic、OpenAI、向量库与爬虫相关包 ==========
# -q 安静安装；包名列表保持原样（即使本 Mock 版未必全部用到）
!pip install -q gradio pydantic openai chromadb sentence-transformers scikit-learn feedparser beautifulsoup4 requests plotly


In [ ]:
# ========== 导入：标准库并发/日志 + Gradio / Plotly / Pydantic ==========

# os：读路径、判断 memory 文件是否存在
import os
# logging：代理与框架的信息日志（稍后会接到队列）
import logging
# queue：线程安全队列，给 UI 拉日志
import queue
# threading：扫描放后台线程，避免卡住 Gradio
import threading
# time：Mock 扫描/估价里的 sleep，以及 UI 轮询间隔
import time
# json：读写 mock_memory.json
import json
# typing：List / Optional 标注返回值
from typing import List, Optional
# datetime：扫描触发时间戳写进日志
from datetime import datetime

# gradio：Blocks UI、表格、Timer、Plot
import gradio as gr
# plotly：3D 散点「向量空间」示意
import plotly.graph_objects as go
# pydantic BaseModel：Deal / Opportunity 等结构化数据
from pydantic import BaseModel


In [ ]:
# ========== 数据模型 + Mock 代理公司 + Framework（记忆/阈值/绘图数据）==========

# Deal：一条优惠的最小字段（描述、价格、链接）
class Deal(BaseModel):
    product_description: str
    price: float
    url: str


# DealSelection：扫描结果包装成 deals 列表（贴近课程结构化输出）
class DealSelection(BaseModel):
    deals: List[Deal]


# Opportunity：在 Deal 上叠加估值与折扣
class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float


class MockAgent:
    # 默认显示名与终端颜色（ANSI）
    name = "Mock Agent"
    color = '\033[37m'
    
    def log(self, message):
        # 统一走 logging，方便 QueueHandler 截获
        logging.info(f"[{self.name}] {message}")


class MockScannerAgent(MockAgent):
    name = "Scanner Agent"
    
    def scan(self, memory=None):
        # 模拟 RSS 扫描：sleep 一下制造「正在抓取」的体感
        self.log("Simulating RSS feed scan")
        time.sleep(1)
        
        # 固定两条示例优惠（描述/价格/URL 字符串保持原文，便于演示）
        deals = [
            Deal(
                product_description="Apple iPad Pro 11-inch 256GB WiFi (latest model) - Space Gray. Features M2 chip, Liquid Retina display, 12MP camera, Face ID, and all-day battery life.",
                price=749.99,
                url="https://example.com/ipad"
            ),
            Deal(
                product_description="Sony WH-1000XM5 Wireless Noise Cancelling Headphones - Industry-leading noise cancellation, exceptional sound quality, 30-hour battery life, comfortable design.",
                price=329.99,
                url="https://example.com/sony-headphones"
            )
        ]
        
        return DealSelection(deals=deals)


class MockEnsembleAgent(MockAgent):
    name = "Ensemble Agent"
    
    def price(self, description: str) -> float:
        # 模拟集成定价模型：按关键词返回「真值」附近的假估值
        self.log(f"Estimating price for product")
        time.sleep(0.5)
        
        if "iPad" in description:
            return 899.00
        elif "Sony" in description:
            return 398.00
        else:
            return 150.00


class MockMessagingAgent(MockAgent):
    name = "Messaging Agent"
    
    def alert(self, opportunity: Opportunity):
        # 模拟推送：只写日志，不调真实通知 API
        self.log(f"Alert sent: ${opportunity.discount:.2f} discount on {opportunity.deal.product_description[:50]}...")


class MockPlanningAgent(MockAgent):
    name = "Planning Agent"
    # 折扣超过该阈值才告警（美元）
    DEAL_THRESHOLD = 50
    
    def __init__(self):
        # 组装虚拟公司：扫描员、集成估价、信使
        self.scanner = MockScannerAgent()
        self.ensemble = MockEnsembleAgent()
        self.messenger = MockMessagingAgent()
    
    def plan(self, memory=None) -> Optional[Opportunity]:
        # 一轮规划：扫描 → 逐条估价 → 按折扣排序 → 超阈值则告警并返回最优
        if memory is None:
            memory = []
        
        self.log("Starting planning cycle")
        
        selection = self.scanner.scan(memory)
        
        if selection and selection.deals:
            opportunities = []
            for deal in selection.deals:
                # 集成代理给出估值
                estimate = self.ensemble.price(deal.product_description)
                # 折扣 = 估值 - 成交价
                discount = estimate - deal.price
                opportunities.append(Opportunity(
                    deal=deal,
                    estimate=estimate,
                    discount=discount
                ))
            
            # 利润从高到低
            opportunities.sort(key=lambda x: x.discount, reverse=True)
            best = opportunities[0]
            
            self.log(f"Best deal has discount: ${best.discount:.2f}")
            
            if best.discount > self.DEAL_THRESHOLD:
                self.messenger.alert(best)
                return best
        
        return None


class MockDealAgentFramework:
    # 持久化「已发现机会」的本地 JSON 文件名
    MEMORY_FILE = "mock_memory.json"
    
    def __init__(self):
        # 启动时读入历史记忆；planner 懒加载
        self.memory = self.read_memory()
        self.planner = None
    
    def init_agents_as_needed(self):
        # 只初始化一次 PlanningAgent
        if not self.planner:
            logging.info("Initializing Mock Agent Framework")
            self.planner = MockPlanningAgent()
            logging.info("Mock Agent Framework ready")
    
    def read_memory(self) -> List[Opportunity]:
        # 从 JSON 还原 Opportunity 列表；失败则空列表
        if os.path.exists(self.MEMORY_FILE):
            try:
                with open(self.MEMORY_FILE, 'r') as f:
                    data = json.load(f)
                return [Opportunity(**item) for item in data]
            except:
                return []
        return []
    
    def write_memory(self):
        # pydantic .dict() 序列化后写回磁盘
        data = [opp.dict() for opp in self.memory]
        with open(self.MEMORY_FILE, 'w') as f:
            json.dump(data, f, indent=2)
    
    def run(self) -> List[Opportunity]:
        # 跑一轮 plan；若有结果则追加记忆并落盘
        self.init_agents_as_needed()
        result = self.planner.plan(memory=self.memory)
        
        if result:
            self.memory.append(result)
            self.write_memory()
        
        return self.memory
    
    @classmethod
    def get_plot_data(cls, max_datapoints=100):
        # 演示用随机 3D 向量（不是真实 embedding）
        import numpy as np
        
        n_points = min(100, max_datapoints)
        vectors = np.random.randn(n_points, 3)
        documents = [f"Product {i}" for i in range(n_points)]
        colors = ['red', 'blue', 'green', 'orange'] * (n_points // 4 + 1)
        
        return documents[:n_points], vectors, colors[:n_points]


In [ ]:
# ========== UI 辅助：ANSI→HTML 颜色、日志队列、Plotly 3D ==========

# 终端 ANSI 颜色码：后面映射成 HTML span 颜色
BG_BLACK = '\033[40m'
RED = '\033[31m'
GREEN = '\033[32m'
YELLOW = '\033[33m'
BLUE = '\033[34m'
MAGENTA = '\033[35m'
CYAN = '\033[36m'
WHITE = '\033[37m'
BG_BLUE = '\033[44m'
RESET = '\033[0m'

# ANSI 前缀 → CSS 颜色；用于把日志渲染进 Gradio HTML
color_mapper = {
    BG_BLACK+RED: "#dd0000",
    BG_BLACK+GREEN: "#00dd00",
    BG_BLACK+YELLOW: "#dddd00",
    BG_BLACK+BLUE: "#0000ee",
    BG_BLACK+MAGENTA: "#aa00dd",
    BG_BLACK+CYAN: "#00dddd",
    BG_BLACK+WHITE: "#87CEEB",
    BG_BLUE+WHITE: "#ff7800"
}


def reformat_log(message):
    # 把 ANSI 颜色替换成 <span style=...>，RESET 换成 </span>
    for key, value in color_mapper.items():
        message = message.replace(key, f'<span style="color: {value}">')
    message = message.replace(RESET, '</span>')
    return message


class QueueHandler(logging.Handler):
    def __init__(self, log_queue):
        # 自定义 Handler：emit 时把格式化后的记录丢进队列
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        # format(record) 得到字符串，put 到线程安全队列
        self.log_queue.put(self.format(record))


def setup_logging(log_queue):
    # 根 logger 挂上 QueueHandler，INFO 级别
    handler = QueueHandler(log_queue)
    formatter = logging.Formatter(
        "[%(asctime)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger = logging.getLogger()
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


def html_for(log_data):
    # 只展示最近 20 条，做成可滚动深色终端风格 div
    output = '<br>'.join(log_data[-20:])
    return f"""
    <div style="height: 420px; overflow-y: auto; border: 1px solid #444; background-color: #1a1a1a; padding: 12px; font-family: monospace; font-size: 13px; color: #fff;">
        {output}
    </div>
    """


def get_plot():
    try:
        # 取演示用随机向量点
        documents, vectors, colors = MockDealAgentFramework.get_plot_data(max_datapoints=100)
        
        # 3D 散点：x/y/z 来自随机向量各列
        fig = go.Figure(data=[go.Scatter3d(
            x=vectors[:, 0],
            y=vectors[:, 1],
            z=vectors[:, 2],
            mode='markers',
            marker=dict(size=3, color=colors, opacity=0.7),
        )])
        
        # 深色主题布局与固定相机角度（title 字符串保持原文）
        fig.update_layout(
            scene=dict(
                xaxis_title='X', 
                yaxis_title='Y', 
                zaxis_title='Z',
                aspectmode='manual',
                aspectratio=dict(x=2.2, y=2.2, z=1),
                camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
                bgcolor='#1a1a1a'
            ),
            height=420,
            margin=dict(r=5, b=5, l=5, t=5),
            paper_bgcolor='#1a1a1a',
            font=dict(color='#ffffff'),
            title="Mock Vector Space (Random Data for Demo)"
        )
        return fig
        
    except Exception as e:
        # 失败时返回带错误标题的空图，避免 UI 崩
        fig = go.Figure()
        fig.update_layout(
            title=f'Error: {str(e)}',
            height=420,
            paper_bgcolor='#1a1a1a',
            font=dict(color='#ffffff')
        )
        return fig


In [ ]:
# ========== DealHunterApp：表格映射、后台扫描、Gradio Blocks 布局 ==========

class DealHunterApp:
    
    def __init__(self):
        # 懒持有 Framework 实例
        self.framework = None
    
    def get_framework(self):
        # 首次调用时创建并 init agents
        if not self.framework:
            self.framework = MockDealAgentFramework()
            self.framework.init_agents_as_needed()
        return self.framework
    
    def opportunities_to_table(self, opportunities):
        # 把 Opportunity 列表转成 Gradio Dataframe 行
        if not opportunities:
            return []
        
        return [
            [
                opp.deal.product_description,
                f"${opp.deal.price:.2f}",
                f"${opp.estimate:.2f}",
                f"${opp.discount:.2f}",
                opp.deal.url
            ]
            for opp in opportunities
            if isinstance(opp, Opportunity)
        ]
    
    def scan_for_deals(self):
        # 同步跑一轮 framework.run，返回表格数据
        framework = self.get_framework()
        logging.info(f"Scan triggered at {datetime.now().strftime('%H:%M:%S')} - Current memory: {len(framework.memory)} deals")
        new_opportunities = framework.run()
        logging.info(f"Scan complete - Total deals: {len(framework.memory)}")
        return self.opportunities_to_table(new_opportunities)
    
    def scan_with_logging(self, log_data):
        # Generator：后台线程扫描，同时从队列拉日志 yield 给 Gradio
        log_queue = queue.Queue()
        result_queue = queue.Queue()
        setup_logging(log_queue)
        
        def worker():
            # 工作线程：跑扫描，结果放进 result_queue
            result = self.scan_for_deals()
            result_queue.put(result)
        
        thread = threading.Thread(target=worker)
        thread.start()
        
        framework = self.get_framework()
        current_table = self.opportunities_to_table(framework.memory)
        
        while True:
            try:
                # 有新日志：追加并刷新 HTML + 表格
                message = log_queue.get_nowait()
                log_data.append(reformat_log(message))
                current_table = self.opportunities_to_table(framework.memory)
                yield log_data, html_for(log_data), current_table
            except queue.Empty:
                try:
                    # 扫描结束：吐出最终表格并 return 结束 generator
                    final_table = result_queue.get_nowait()
                    yield log_data, html_for(log_data), final_table
                    return
                except queue.Empty:
                    # 两边都空：短暂 sleep 再轮询，避免忙等
                    current_table = self.opportunities_to_table(framework.memory)
                    yield log_data, html_for(log_data), current_table
                    time.sleep(0.1)
    
    def handle_selection(self, selected_index: gr.SelectData):
        # 用户点表格某行：对该机会再发一次 alert
        framework = self.get_framework()
        row = selected_index.index[0]
        
        if row < len(framework.memory):
            opportunity = framework.memory[row]
            framework.planner.messenger.alert(opportunity)
            return f"Alert sent for: {opportunity.deal.product_description[:60]}..."
        
        return "Invalid selection"
    
    def load_initial_state(self):
        # 页面 load：空日志 + 当前记忆表格
        framework = self.get_framework()
        initial_table = self.opportunities_to_table(framework.memory)
        return [], "", initial_table
    
    def launch(self):
        # 搭建 Gradio Blocks：标题、表格、日志、向量图、Timer、行选择
        with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
            
            # 跨回调共享的日志列表状态
            log_data = gr.State([])
            
            # 居中标题 HTML（文案保持原文）
            gr.Markdown(
                '<div style="text-align: center; font-size: 28px; font-weight: bold; margin: 20px 0;">The Price is Right - Demo</div>'
                '<div style="text-align: center; font-size: 16px; color: #666; margin-bottom: 20px;">Multi-Agent Deal Hunting System (Mock Version)</div>'
            )
            
            with gr.Row():
                # 机会表：描述 / 价 / 估值 / 折扣 / URL
                opportunities_table = gr.Dataframe(
                    headers=["Product Description", "Price", "Estimate", "Discount", "URL"],
                    wrap=True,
                    column_widths=[6, 1, 1, 1, 3],
                    row_count=10,
                    col_count=5,
                    max_height=420,
                    interactive=False
                )
            
            with gr.Row():
                with gr.Column(scale=1):
                    # 代理日志 HTML 区
                    logs_display = gr.HTML(label="Agent Logs")
                with gr.Column(scale=1):
                    # 右侧 3D 向量示意
                    vector_plot = gr.Plot(value=get_plot(), show_label=False)
            
            # 首次加载初始状态
            ui.load(
                self.load_initial_state,
                inputs=[],
                outputs=[log_data, logs_display, opportunities_table]
            )
            
            # 每 10 秒触发一次带日志的扫描 generator
            timer = gr.Timer(value=10, active=True)
            timer.tick(
                self.scan_with_logging,
                inputs=[log_data],
                outputs=[log_data, logs_display, opportunities_table]
            )
            
            # 隐藏文本框承接选中行的反馈
            selection_feedback = gr.Textbox(visible=False)
            opportunities_table.select(
                self.handle_selection,
                inputs=[],
                outputs=[selection_feedback]
            )
        
        # share=True 生成公网链接；inbrowser 尝试自动打开浏览器
        ui.launch(share=True, inbrowser=True)


In [ ]:
# ========== 入口：实例化 App 并启动 Gradio ==========

# 创建应用对象（内部懒加载 Mock Framework）
app = DealHunterApp()
# 弹出/分享 Gradio UI（阻塞直到关闭）
app.launch()
